# Construindo o pipeline do zero: do CSV ao DuckDB

Este notebook é uma aula guiada: você vai construir, célula por célula, as mesmas peças que já existem em `scripts/extract_csv.py` e `scripts/extract_weather.py`, só que aqui dá pra parar em cada passo e entender o "porquê" antes de ver a versão final do script.

**Importante:** este notebook grava num banco **separado** (`warehouse/aula_pipeline.duckdb`), não no `workshop.duckdb` que alimenta o Metabase: pode rodar, quebrar e rodar de novo sem medo de estragar a dashboard da aula.

Roteiro:
1. Ler um CSV
2. O problema: duas fontes, dois esquemas
3. Renomear colunas
4. Transformar isso numa função reutilizável
5. Escrever o resultado no DuckDB
6. Fazer uma requisição numa API externa
7. Juntar tudo num mini pipeline, o mesmo desenho do `run_pipeline.py`

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import requests

RAW_CSV = Path("../data/raw/vendas_lojas.csv")
INCOMING_CSV = Path("../data/incoming/vendas_lojas_recentes.csv")

# Banco de sandbox só desta aula, separado do warehouse/workshop.duckdb
# que o Metabase usa, pra podermos experimentar sem medo de estragar nada
DB_PATH = Path("../warehouse/aula_pipeline.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Lendo um CSV

`pandas.read_csv` já lê pelo **nome** da coluna (o header), não pela posição. Isso importa porque, como vamos ver já na próxima seção, as duas fontes de venda da GeloDados têm colunas em nomes e ordens diferentes: ler por posição quebraria silenciosamente.

In [ ]:
historico = pd.read_csv(RAW_CSV)
historico.head()


In [ ]:
print(f"{len(historico)} linhas")
print(f"colunas: {list(historico.columns)}")


## 2. O problema: duas fontes, dois esquemas

O time comercial trocou de sistema de ponto de venda na metade da janela de dados. O arquivo `vendas_lojas_recentes.csv` tem **a mesma informação**, só que com colunas renomeadas e em outra ordem.


In [ ]:
recentes = pd.read_csv(INCOMING_CSV)
recentes.head()


In [ ]:
# As colunas não batem: juntar os dois DataFrames direto (pd.concat) ia
# criar um Frankenstein de colunas, a maioria cheia de NaN
set(historico.columns) ^ set(recentes.columns)

## 3. Renomeando colunas

A solução: um dicionário explícito de `nome_na_fonte -> nome_canônico`, igual ao que `scripts/transform.sql` faz em SQL. Aqui fazemos a mesma reconciliação, só que em pandas.


In [ ]:
mapeamento_recentes = {
    "dt_venda": "data_venda",
    "id_loja": "loja_id",
    "uf_cidade": "cidade",
    "id_cliente": "cliente_id",
    "nm_cliente": "nome_cliente",
    "idade": "idade_cliente",
    "genero": "genero_cliente",
    "categoria_produto": "produto",
    "sabor_item": "sabor",
    "qtd_vendida": "unidades_vendidas",
    "vl_unitario": "preco_unitario",
    "vl_total": "receita_brl",
    "tipo_pagamento": "forma_pagamento",
}

recentes_renomeado = recentes.rename(columns=mapeamento_recentes)
recentes_renomeado.head()


In [ ]:
assert set(historico.columns) == set(recentes_renomeado.columns)

vendas = pd.concat([historico, recentes_renomeado], ignore_index=True)
print(f"{len(vendas)} linhas no total, agora com um único esquema de colunas")


## 4. Transformando isso numa função

Reparou que "ler o CSV" e "renomear se precisar" é uma receita que vamos repetir para qualquer fonte nova que aparecer? Isso é sinal de que vale a pena virar uma função.


In [ ]:
def carregar_vendas(caminho_csv, mapeamento=None):
    df = pd.read_csv(caminho_csv)
    if mapeamento:
        df = df.rename(columns=mapeamento)
    return df


historico_v2 = carregar_vendas(RAW_CSV)
recentes_v2 = carregar_vendas(INCOMING_CSV, mapeamento=mapeamento_recentes)

vendas = pd.concat([historico_v2, recentes_v2], ignore_index=True)
vendas.sample(5)


## 5. Escrevendo no DuckDB

DuckDB não tem um "insira este DataFrame direto numa tabela nova": o jeito mais simples é **registrar** o DataFrame como uma view temporária (`con.register`) e usar `CREATE TABLE ... AS SELECT * FROM` para materializar.

Também usamos **schemas** (`raw`, `staging`, `mart`) pra deixar claro em que estágio de limpeza cada tabela está. Aqui só vamos até `raw`.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

con.register("_tmp_vendas", vendas)
con.execute("CREATE OR REPLACE TABLE raw.vendas AS SELECT * FROM _tmp_vendas")
con.unregister("_tmp_vendas")

con.execute("SELECT COUNT(*) AS total FROM raw.vendas").df()


Essa sequência (`register` → `CREATE OR REPLACE TABLE` → `unregister`) também vale a pena virar função: vamos usá-la de novo já na próxima seção.

In [ ]:
def salvar_no_duckdb(con, df, tabela):
    con.register("_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {tabela} AS SELECT * FROM _tmp")
    con.unregister("_tmp")
    print(f"{len(df)} linhas -> {tabela}")


salvar_no_duckdb(con, historico_v2, "raw.vendas_historico")
salvar_no_duckdb(con, recentes_v2, "raw.vendas_recentes")


## 6. Fazendo uma requisição numa API

O clima não existe nos sistemas internos da GeloDados: precisa vir de uma **API pública**, a [Open-Meteo](https://open-meteo.com/). `requests.get` com os parâmetros certos (coordenadas + janela de datas) devolve um JSON com a série diária.

In [ ]:
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": -23.5505,
        "longitude": -46.6333,
        "start_date": "2026-06-01",
        "end_date": "2026-06-10",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "America/Sao_Paulo",
    },
    timeout=10,
)
resp.raise_for_status()
dados = resp.json()
dados["daily"]


O JSON vem "achatado" em listas paralelas (uma lista de datas, uma de temperaturas máximas, etc.). É só remontar isso num DataFrame.

In [ ]:
clima_sp = pd.DataFrame({
    "cidade": "Sao Paulo",
    "data": dados["daily"]["time"],
    "temp_max_c": dados["daily"]["temperature_2m_max"],
    "temp_min_c": dados["daily"]["temperature_2m_min"],
    "precipitacao_mm": dados["daily"]["precipitation_sum"],
})
clima_sp


In [ ]:
# mesma função de sempre, não importa se o dado veio de CSV ou de API
salvar_no_duckdb(con, clima_sp, "raw.clima_sp")

## 7. Juntando tudo: o mesmo desenho do `run_pipeline.py`

Com as peças prontas (`carregar_vendas`, `salvar_no_duckdb`), "o pipeline" é só chamar essas funções na ordem certa. Dá uma olhada em `scripts/run_pipeline.py` depois: é exatamente essa ideia, só chamando os scripts de verdade em vez de funções dentro de um notebook.

In [ ]:
def pipeline_de_vendas():
    historico = carregar_vendas(RAW_CSV)
    recentes = carregar_vendas(INCOMING_CSV, mapeamento=mapeamento_recentes)
    salvar_no_duckdb(con, historico, "raw.vendas_historico")
    salvar_no_duckdb(con, recentes, "raw.vendas_recentes")


pipeline_de_vendas()


In [ ]:
con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""").df()


In [ ]:
con.close()


## Para ir além

- Compare o que você fez aqui com `scripts/extract_csv.py` e `scripts/extract_weather.py`: a lógica é a mesma; os scripts só têm mais tratamento de erro (cache offline, fallback de coordenadas se a API de geocoding falhar).
- `scripts/transform.sql` é o próximo andar do pipeline: pega essas tabelas `raw.*` e transforma em `stg_*`/`mart_*` com SQL puro, fica pra outra aula.
- **Exercício:** repita a seção 6 para outra cidade (ex.: Curitiba, lat `-25.4284`, lon `-49.2733`) e salve como `raw.clima_curitiba`.
- Quando estiver confiante, rode o pipeline de verdade: `python scripts/run_pipeline.py`. Ele usa o banco oficial `workshop.duckdb` (não o banco de sandbox que criamos aqui) e já inclui a camada de staging/mart.